In [ ]:
import pandas as pd

# Load restaurant dataset
restaurants = pd.read_csv(
    "data/raw/Restaurant names and Metadata.csv"
)

# Load customer reviews dataset
reviews = pd.read_csv(
    "data/raw/Restaurant reviews_public.csv"
)

print("Datasets loaded successfully!")

In [ ]:
restaurants.head()

In [ ]:
reviews.head()

In [ ]:
restaurants.shape()


In [ ]:
restaurants.shape

In [ ]:
reviews.shape

In [ ]:
restaurants.info()


In [ ]:
reviews.info

In [ ]:
restaurants.isnull().sum()

In [ ]:
reviews.isnull().sum()

In [ ]:
restaurants.duplicated().sum()

In [ ]:
reviews.duplicated().sum()

In [ ]:
print("RESTAURANT DATA TYPES")
print(restaurants.dtypes)

print("\nREVIEW DATA TYPES")
print(reviews.dtypes)

In [ ]:
reviews["Rating"].value_counts(dropna=False)

In [ ]:
reviews["Review"].astype("string").str.strip().eq("").sum()

In [ ]:
restaurant_names = set(
    restaurants["Name"].str.strip().str.casefold()
)

review_names = set(
    reviews["Restaurant"].str.strip().str.casefold()
)

print("Restaurants in metadata:", len(restaurant_names))
print("Restaurants in reviews:", len(review_names))

print(
    "Reviews restaurants missing from metadata:",
    review_names - restaurant_names
)

In [ ]:
rating_numeric = pd.to_numeric(
    reviews["Rating"],
    errors="coerce"
)

print("Original ratings:", reviews["Rating"].nunique())

print("Ratings that cannot be converted:")
print(
    reviews.loc[
        rating_numeric.isna() & reviews["Rating"].notna(),
        "Rating"
    ].unique()
)

In [ ]:
restaurants_without_reviews = (
    restaurant_names - review_names
)

print("Restaurants without reviews:")
print(restaurants_without_reviews)

In [ ]:
print("Minimum rating:", rating_numeric.min())
print("Maximum rating:", rating_numeric.max())

print("Missing or invalid ratings:", rating_numeric.isna().sum())

In [ ]:
restaurants["Cost"].unique()

In [ ]:
rating_numeric.value_counts().sort_index()

In [ ]:
print("Restaurant missing values:")
print(restaurants.isnull().sum())

print("\nReview missing values:")
print(reviews.isnull().sum())

In [ ]:
restaurants["Cost"].unique()

In [ ]:
rating_numeric.value_counts().sort_index()

In [ ]:
print("Restaurant missing values:")
print(restaurants.isnull().sum())

print("\nReview missing values:")
print(reviews.isnull().sum())

In [ ]:
# Create copies of original datasets
restaurants_clean = restaurants.copy()
reviews_clean = reviews.copy()

print("Clean copies created successfully!")

In [ ]:
# Remove commas from cost and convert to numeric
restaurants_clean["Cost"] = pd.to_numeric(
    restaurants_clean["Cost"].str.replace(",", "", regex=False),
    errors="coerce"
)

print(restaurants_clean["Cost"].dtype)
print(restaurants_clean["Cost"].head())

In [ ]:
# Convert ratings into numeric values
reviews_clean["Rating"] = pd.to_numeric(
    reviews_clean["Rating"],
    errors="coerce"
)

print(reviews_clean["Rating"].dtype)
print(reviews_clean["Rating"].isnull().sum())

In [ ]:
# Count duplicates before removal
print("Duplicates before cleaning:",
      reviews_clean.duplicated().sum())

# Remove exact duplicate rows
reviews_clean = reviews_clean.drop_duplicates()

# Check again
print("Duplicates after cleaning:",
      reviews_clean.duplicated().sum())

In [ ]:
# Keep only rows containing actual review text
sentiment_data = reviews_clean.dropna(
    subset=["Review"]
).copy()

print("Reviews available for sentiment analysis:",
      len(sentiment_data))

print("Missing review text:",
      sentiment_data["Review"].isnull().sum())

In [ ]:
from pathlib import Path

# Make sure the processed folder exists
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

# Save cleaned datasets
restaurants_clean.to_csv(
    "data/processed/restaurants_clean.csv",
    index=False
)

reviews_clean.to_csv(
    "data/processed/reviews_clean.csv",
    index=False
)

sentiment_data.to_csv(
    "data/processed/sentiment_data.csv",
    index=False
)

print("All cleaned datasets saved successfully!")

In [ ]:
print("Restaurant dataset:", restaurants_clean.shape)
print("Review dataset:", reviews_clean.shape)
print("Sentiment dataset:", sentiment_data.shape)

print("\nRestaurant cost type:")
print(restaurants_clean["Cost"].dtype)

print("\nRating type:")
print(reviews_clean["Rating"].dtype)

print("\nRemaining duplicate reviews:")
print(reviews_clean.duplicated().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a consistent chart style
sns.set_theme(style="whitegrid")

print("Visualization libraries imported successfully!")

In [ ]:
print("Restaurant Cost Statistics")
print(restaurants_clean["Cost"].describe())

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=restaurants_clean,
    x="Cost",
    bins=15,
    kde=True
)

plt.title("Distribution of Restaurant Costs")
plt.xlabel("Restaurant Cost")
plt.ylabel("Number of Restaurants")

plt.tight_layout()
plt.show()

In [ ]:
# Split comma-separated cuisines into individual values
cuisine_series = (
    restaurants_clean["Cuisines"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

# Count each cuisine
cuisine_counts = cuisine_series.value_counts()

print("Top 10 cuisines:")
print(cuisine_counts.head(10))

In [ ]:
plt.figure(figsize=(11, 6))

sns.barplot(
    x=cuisine_counts.head(10).values,
    y=cuisine_counts.head(10).index
)

plt.title("Top 10 Most Common Cuisines")
plt.xlabel("Number of Restaurants")
plt.ylabel("Cuisine")

plt.tight_layout()
plt.show()

In [ ]:
rating_counts = reviews_clean["Rating"].value_counts().sort_index()

print(rating_counts)

In [ ]:
plt.figure(figsize=(9, 5))

sns.countplot(
    data=reviews_clean,
    x="Rating",
    order=sorted(reviews_clean["Rating"].dropna().unique())
)

plt.title("Distribution of Customer Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("EDA setup completed!")

In [ ]:
restaurants_clean["Cost"].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=restaurants_clean,
    x="Cost",
    bins=15,
    kde=True
)

plt.title("Distribution of Restaurant Costs")
plt.xlabel("Restaurant Cost")
plt.ylabel("Number of Restaurants")

plt.tight_layout()
plt.show()

In [ ]:
# Convert ratings into numeric values
reviews_clean["Rating"] = pd.to_numeric(
    reviews_clean["Rating"],
    errors="coerce"
)

# Calculate average rating for each restaurant
restaurant_ratings = (
    reviews_clean
    .groupby("Restaurant")["Rating"]
    .mean()
    .sort_values(ascending=False)
)

print("Top 10 highest-rated restaurants:")
print(restaurant_ratings.head(10))

print("\nOverall average rating:")
print(reviews_clean["Rating"].mean())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 5))

sns.histplot(
    data=reviews_clean,
    x="Rating",
    bins=10,
    kde=True
)

plt.title("Distribution of Customer Ratings")
plt.xlabel("Customer Rating")
plt.ylabel("Number of Reviews")

plt.tight_layout()
plt.show()

In [ ]:
# Create a copy of the cleaned restaurant dataset
restaurant_features = restaurants_clean.copy()

# Count the number of cuisines offered by each restaurant
restaurant_features["Cuisine_Count"] = (
    restaurant_features["Cuisines"]
    .str.split(",")
    .str.len()
)

# Calculate average rating and total reviews for each restaurant
rating_summary = (
    reviews_clean
    .groupby("Restaurant")["Rating"]
    .agg(
        Average_Rating="mean",
        Total_Reviews="count"
    )
    .reset_index()
)

# Merge restaurant information with rating information
restaurant_features = restaurant_features.merge(
    rating_summary,
    left_on="Name",
    right_on="Restaurant",
    how="left"
)

# Remove the extra Restaurant column
restaurant_features.drop(columns=["Restaurant"], inplace=True)

# Display the new features
restaurant_features.head()

In [ ]:
print("New dataset shape:", restaurant_features.shape)

print("\nNew columns:")
print(restaurant_features.columns.tolist())

print("\nMissing values:")
print(restaurant_features.isnull().sum())

In [ ]:
restaurant_features.to_csv(
    "data/processed/restaurant_features.csv",
    index=False
)

print("Feature-engineered dataset saved successfully!")